In [3]:
import pandas as pd
import numpy as np

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
import os

warnings.filterwarnings('ignore')

os.makedirs('../plots', exist_ok=True)

# Load dataset
df = pd.read_csv('../data/student_loans.csv')

# Create figure
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

fig.suptitle(
    'Student Loan Early Warning - EDA',
    fontsize=14,
    fontweight='bold'
)

# ---------------------------------------------------
# Chart 1: Default rate across 3 horizons
# ---------------------------------------------------

horizons = ['30-Day', '60-Day', '90-Day']

default_rates = [
    df['default_30d'].mean() * 100,
    df['default_60d'].mean() * 100,
    df['default_90d'].mean() * 100
]

bars = axes[0, 0].bar(
    horizons,
    default_rates,
    color=['#16A34A', '#EAB308', '#EF4444'],
    edgecolor='black',
    alpha=0.85
)

axes[0, 0].set_title(
    'Default Rate by Time Horizon',
    fontweight='bold'
)

axes[0, 0].set_ylabel('Default Rate (%)')

for bar, val in zip(bars, default_rates):
    axes[0, 0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.1,
        f'{val:.1f}%',
        ha='center',
        fontweight='bold',
        fontsize=12
    )

# ---------------------------------------------------
# Chart 2: Default rate by employment
# ---------------------------------------------------

emp_def = df.groupby('employed')['default_90d'].mean() * 100

axes[0, 1].bar(
    ['Unemployed', 'Employed'],
    [emp_def[0], emp_def[1]],
    color=['#EF4444', '#16A34A'],
    edgecolor='black',
    alpha=0.85
)

axes[0, 1].set_title(
    '90-Day Default: Employed vs Unemployed',
    fontweight='bold'
)

axes[0, 1].set_ylabel('Default Rate (%)')

# ---------------------------------------------------
# Chart 3: Default by GPA bucket
# ---------------------------------------------------

df['gpa_bucket'] = pd.cut(
    df['gpa'],
    bins=[5, 6.5, 7.5, 8.5, 10],
    labels=['<6.5', '6.5-7.5', '7.5-8.5', '8.5+']
)

gpa_def = df.groupby(
    'gpa_bucket',
    observed=True
)['default_90d'].mean() * 100

colors_g = ['#EF4444', '#F97316', '#EAB308', '#16A34A']

axes[0, 2].bar(
    gpa_def.index,
    gpa_def.values,
    color=colors_g,
    edgecolor='black',
    alpha=0.85
)

axes[0, 2].set_title(
    '90-Day Default Rate by GPA',
    fontweight='bold'
)

# ---------------------------------------------------
# Chart 4: EMI/Income ratio distribution
# ---------------------------------------------------

axes[1, 0].hist(
    df[df['default_90d'] == 0]['emi_income'].clip(0, 1),
    bins=50,
    alpha=0.6,
    color='green',
    label='No Default',
    density=True
)

axes[1, 0].hist(
    df[df['default_90d'] == 1]['emi_income'].clip(0, 1),
    bins=50,
    alpha=0.6,
    color='red',
    label='Default',
    density=True
)

axes[1, 0].axvline(
    0.45,
    color='black',
    linestyle='--',
    linewidth=2
)

axes[1, 0].set_title(
    'EMI/Income: Default vs No Default',
    fontweight='bold'
)

axes[1, 0].legend()

# ---------------------------------------------------
# Chart 5: Default by institute tier
# ---------------------------------------------------

tier_def = df.groupby('institute_tier')['default_90d'].mean() * 100

axes[1, 1].bar(
    tier_def.index,
    tier_def.values,
    color=['#1D4ED8', '#7C3AED', '#EF4444'],
    edgecolor='black',
    alpha=0.85
)

axes[1, 1].set_title(
    'Default Rate by Institute Tier',
    fontweight='bold'
)

# ---------------------------------------------------
# Chart 6: Vintage analysis
# ---------------------------------------------------

df['vintage_bucket'] = pd.cut(
    df['months_since_disb'],
    bins=[0, 6, 12, 24, 36, 60],
    labels=['0-6m', '6-12m', '1-2yr', '2-3yr', '3-5yr']
)

vintage_def = df.groupby(
    'vintage_bucket',
    observed=True
)['default_90d'].mean() * 100

axes[1, 2].bar(
    vintage_def.index,
    vintage_def.values,
    color='#7C3AED',
    edgecolor='black',
    alpha=0.85
)

axes[1, 2].set_title(
    'Default Rate by Loan Vintage',
    fontweight='bold'
)

# ---------------------------------------------------
# Save plot
# ---------------------------------------------------

plt.tight_layout()

plt.savefig(
    '../plots/student_loan_eda.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("Saved: plots/student_loan_eda.png")

print()
print("YOUR 3 INTERVIEW INSIGHTS:")

unemp = df[df['employed'] == 0]['default_90d'].mean() * 100
emp_r = df[df['employed'] == 1]['default_90d'].mean() * 100

print(
    f"1. Unemployed students default at "
    f"{unemp:.1f}% vs {emp_r:.1f}% for employed"
)

low_gpa = df[df['gpa'] < 6.5]['default_90d'].mean() * 100
hi_gpa = df[df['gpa'] > 8.5]['default_90d'].mean() * 100

print(
    f"2. GPA<6.5 has {low_gpa:.1f}% default "
    f"vs {hi_gpa:.1f}% for GPA>8.5"
)

early = df[df['months_since_disb'] <= 6]['default_90d'].mean() * 100

print(
    f"3. First 6 months after moratorium: "
    f"{early:.1f}% default - highest risk window"
)

Saved: plots/student_loan_eda.png

YOUR 3 INTERVIEW INSIGHTS:
1. Unemployed students default at 54.7% vs 22.2% for employed
2. GPA<6.5 has 45.7% default vs 26.8% for GPA>8.5
3. First 6 months after moratorium: 39.7% default - highest risk window
